# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

# Import Functions
sys.path.append("../../")

from matplotlib import pyplot as plt
from matplotlib import pyplot as plt

from os.path import join, exists
import sys
sys.path.append("../") # Add directory containing src/data to path

from src.models.postnet.posterior_networks.run import run
from src.models.postnet.posterior_networks.run_eval import run_eval
from src.configs.octmnist_config import data_name, data_name_oods, postnet_param
from src.file_manager.filepath import FilePath
from src.models.postnet.posterior_networks.PosteriorNetwork import PosteriorNetwork
from src.configs.default_configs import fn_model, fn_pred
from src.models.postnet.result_processing import process_pn_results
from src.file_manager.load_save_df import load_pred_df, save_pred_perf_df
from src.evaluation.evaluate import get_model_performance

from src.data_generator.oct_mnist import load_octmnist_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_generator.octdl import load_octdl_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets

# from cur_seed import seed
seed = 2028

fp = FilePath(data_name=data_name, seed=seed)
fp_ood = FilePath(data_name=data_name_oods[0], seed=seed)
fp_ood2 = FilePath(data_name=data_name_oods[1], seed=seed)

directory_model=fp.get_parent_folder(folder_name=fn_model)
directory_results=fp.get_parent_folder(folder_name=fn_pred)
fn_model = f"model-dpn-{seed}-{data_name}-{seed}-conv-[224, 224, 3]-4"

# Load Data

In [ ]:
data_dict = load_octmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])
data_dict_ood = load_chestmnist_data_dict(fp_preprocessed=fp_ood.get_preprocessed_folder(), only_test=True)
data_dict_ood = process_dataset_for_ood(data_dict, data_dict_ood, seed)
octdl_in_data_dict, octdl_out_data_dict = load_octdl_data_dict(
    fp_preprocessed=fp_ood2.get_preprocessed_folder())
data_dict = left_join_datasets(data_dict, octdl_in_data_dict)
octdl_in_data_dict = process_dataset_for_ood(data_dict, octdl_in_data_dict, seed)
octdl_out_data_dict = process_dataset_for_ood(data_dict, octdl_out_data_dict, seed)

# Training

In [ ]:
if not exists(join(directory_model, fn_model)):
    results_metrics = run(
        data_dict=data_dict,
        # Directory
        directory_model=directory_model,
        directory_results=directory_results,
        # Seeds
        seed_dataset=seed, # No shuffling
        seed_model=seed,
        **postnet_param
    )

# Prediction

In [ ]:
ood_postnet_param = postnet_param.copy()
ood_data_dicts = {
    "ood_in_octdl": octdl_in_data_dict, 
    "ood_out_octdl": octdl_out_data_dict, 
    "ood_chestmnist": data_dict_ood
}
fp_results = run_eval(
    data_dict=data_dict,
    ood_data_dicts = ood_data_dicts,
    # Directory
    directory_model=directory_model,
    directory_results=directory_results,
    # Seeds
    seed_dataset=seed, # No shuffling
    seed_model=seed,
    **ood_postnet_param,
    fn_model=fn_model,
)

# Process Output

In [ ]:
ood_data_dicts = {
    "ood_in_octdl": octdl_in_data_dict, 
    "ood_out_octdl": octdl_out_data_dict, 
    "ood_chestmnist": data_dict_ood
}
process_pn_results(data_dict, ood_data_dicts, directory_results, fp)

# Evaluate Pred Perf

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=PosteriorNetwork)
pred_df["split_perf"] = pred_df["split"]
pred_df["split_perf"][pred_df["split"]=="Test"] = ["Test-OCTMNIST" for i in range(num_ori_test)] + \
    ["Test-OCTDL" for i in range((pred_df["split"]=="Test").sum()-num_ori_test)]
perf_df = get_model_performance(all_pred_df=pred_df, data_dict=data_dict, label="pn", perf_split_col="split_perf")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=PosteriorNetwork)
perf_df